# Running MEEP Simulations

[MEEP](https://meep.readthedocs.io/) is an open-source FDTD electromagnetic simulator. This notebook demonstrates using the `gsim.meep` API to run an S-parameter simulation on a photonic Y-branch.

**Requirements:**

- UBC PDK: `uv pip install ubcpdk`
- [GDSFactory+](https://gdsfactory.com) account for cloud simulation

### Load a pcell from UBC PDK

In [ ]:
from ubcpdk import PDK, cells

PDK.activate()

c = cells.ebeam_y_1550()
c

### Configure and run simulation

In [ ]:
from gsim import meep
from gsim.common.stack import get_stack
from gsim.meep.models.api import Material

stack = get_stack()  # auto-detects active PDK

sim = meep.Simulation()

sim.geometry(component=c, stack=stack)
sim.materials = {
    "si": Material(refractive_index=3.47),
    "SiO2": Material(refractive_index=1.44),
}
sim.source(port="o1", wavelength=1.55, wavelength_span=0.01)
sim.monitors = ["o1", "o2", "o3"]
sim.domain(pml=1.0, margin_x=0.5, margin_y=0.5, z_ref="stack")
sim.solver(resolution=20, simplify_tol=0.01, save_animation=True, verbose_interval=5.0)
sim.solver.stop_when_energy_decayed()

print(sim.validate_config())

In [ ]:
sim.plot_2d(slices="xyz")

### Run simulation on cloud

In [ ]:
# Run on GDSFactory+ cloud
result = sim.run(check_cache=True)

In [ ]:
result.plot_interactive()

In [ ]:
result.plot_interactive(phase=True)

In [ ]:
result.show_animation()